# 03_event_detection

## Threshold Sensitivity

Load the aligned table and see how the number of events changes with the Dst threshold and the merge gap.

In [1]:
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

pd.set_option("display.max_columns", 50)
pd.set_option("display.width", 200)

PROCESSED = Path("../data/processed")
EVENTS = Path("../data/events")

aligned = pd.read_parquet(PROCESSED / "aligned_hourly.parquet")
print("aligned:", aligned.shape, "|", aligned.index.min(), "to", aligned.index.max())

def episodes(mask, merge_gap):
    """Runs of True in `mask`; runs separated by fewer than `merge_gap` hours are merged.
    Returns a list of [start_position, end_position_exclusive]."""
    arr = mask.to_numpy().astype(np.int8)
    e = np.diff(np.r_[0, arr, 0])
    starts, ends = np.flatnonzero(e == 1), np.flatnonzero(e == -1)
    merged = []
    for s, en in zip(starts, ends):
        if merged and s - merged[-1][1] < merge_gap:
            merged[-1][1] = en
        else:
            merged.append([s, en])
    return merged

rows = []
for thr in [-30, -50, -100, -250]:
    for gap in [1, 6, 12, 24]:                      # gap = 1 means no merging
        ep = episodes(aligned["dst"] <= thr, gap)
        dur = [en - s for s, en in ep]
        rows.append({"dst_thr": thr, "merge_gap_h": gap, "events": len(ep),
                     "median_dur_h": float(np.median(dur)), "max_dur_h": max(dur)})
print()
print(pd.DataFrame(rows).to_string(index=False))

aligned: (87024, 20) | 2016-07-26 00:00:00 to 2026-06-29 23:00:00

 dst_thr  merge_gap_h  events  median_dur_h  max_dur_h
     -30            1    1108           3.0        136
     -30            6     679           6.0        217
     -30           12     503          10.0        263
     -30           24     373          19.0        494
     -50            1     351           3.0         73
     -50            6     228           6.0        122
     -50           12     202           7.0        122
     -50           24     175          10.0        138
    -100            1      48           3.0         44
    -100            6      36           7.5         45
    -100           12      33           9.0         45
    -100           24      31          10.0         49
    -250            1       3           9.0         20
    -250            6       2          15.5         20
    -250           12       2          15.5         20
    -250           24       2          15.5         2

Build the storm event table: the core, the window, the class and the DSCOVR coverage for each storm.

In [2]:
THR, GAP, LOOKBACK, REC_LEVEL, REC_CAP = -50, 12, 12, -30, 72

idx = aligned.index
dst = aligned["dst"].to_numpy()
n = len(dst)
ep = episodes(aligned["dst"] <= THR, GAP)

rows = []
for s, en in ep:
    core = dst[s:en]
    k = s + int(np.argmin(core))
    j = en
    while j < n and j < en + REC_CAP and dst[j] <= REC_LEVEL:
        j += 1
    rows.append({"core_start": idx[s], "core_end": idx[en - 1], "min_dst": int(core.min()), "min_time": idx[k],
                 "win_start": idx[max(s - LOOKBACK, 0)], "win_end": idx[min(j, n - 1)]})
ev = pd.DataFrame(rows)

ev["cls"] = pd.cut(ev["min_dst"], [-1000, -250, -100, -49], labels=["super", "intense", "moderate"])
ev["win_hours"] = ((ev["win_end"] - ev["win_start"]) / pd.Timedelta(hours=1) + 1).astype(int)

fld_ok = (aligned["bz_src"] == "dscovr")
pla_ok = (aligned["plasma_src"] == "dscovr")
ev["field_dscovr_pct"] = [round(fld_ok.loc[a:b].mean() * 100) for a, b in zip(ev["win_start"], ev["win_end"])]
ev["plasma_dscovr_pct"] = [round(pla_ok.loc[a:b].mean() * 100) for a, b in zip(ev["win_start"], ev["win_end"])]

print("events:", len(ev))
print(ev["cls"].value_counts().reindex(["moderate", "intense", "super"]).to_string())
print()
print("window length (hours):")
print(ev["win_hours"].describe().round(0).to_string())
print()
print("windows that overlap the next event's window:", int((ev["win_start"].shift(-1) <= ev["win_end"]).sum()))
print()
print("intense and super events:")
print(ev.loc[ev["cls"].isin(["super", "intense"]).to_numpy(),
             ["min_time", "min_dst", "cls", "win_hours", "field_dscovr_pct", "plasma_dscovr_pct"]]
        .sort_values("min_dst").to_string(index=False))

events: 202
cls
moderate    171
intense      29
super         2

window length (hours):
count    202.0
mean      38.0
std       23.0
min       15.0
25%       23.0
50%       30.0
75%       46.0
max      148.0

windows that overlap the next event's window: 29

intense and super events:
           min_time  min_dst     cls  win_hours  field_dscovr_pct  plasma_dscovr_pct
2024-05-11 02:00:00     -406   super        143               100                 29
2024-10-11 01:00:00     -333   super         76                82                 30
2026-01-20 16:00:00     -236 intense        143                73                 33
2025-11-12 04:00:00     -217 intense         71                85                  8
2023-04-24 05:00:00     -213 intense         81               100                 63
2025-01-01 15:00:00     -212 intense         68                72                 71
2024-08-12 16:00:00     -188 intense         70               100                 97
2018-08-26 07:00:00     -176 intens

Find the 48-hour quiet windows that meet my conditions.

In [3]:
W = 48
kp_s, dst_s = aligned["kp"], aligned["dst"]
fld_dsc = (aligned["bz_src"] == "dscovr")
pla_any = (aligned["plasma_src"] != "missing")

excl = pd.Series(False, index=idx)
for a, b in zip(ev["win_start"], ev["win_end"]):
    excl.loc[a - pd.Timedelta(hours=24): b + pd.Timedelta(hours=24)] = True

cand = []
for i in range(0, len(idx) - W + 1, W):
    sl = slice(i, i + W)
    d, k = dst_s.iloc[sl], kp_s.iloc[sl]
    if d.min() > -30 and k.max() <= 3 and fld_dsc.iloc[sl].mean() >= 0.90 \
            and pla_any.iloc[sl].mean() >= 0.95 and not excl.iloc[sl].any():
        cand.append({"start": idx[i], "end": idx[i + W - 1], "min_dst": int(d.min()),
                     "max_kp": float(k.max()), "f107": round(aligned["f107_adj"].iloc[sl].mean()),
                     "plasma_dscovr_pct": round((aligned["plasma_src"].iloc[sl] == "dscovr").mean() * 100)})
q = pd.DataFrame(cand)

print("48-hour blocks in the record:", len(range(0, len(idx) - W + 1, W)))
print("quiet candidates:", len(q))
print()
print("candidates by year:")
print(q.groupby(q["start"].dt.year).size().to_string())
print()
print("by F10.7 band (mean over the window):")
print(pd.cut(q["f107"], [0, 100, 150, 1000], labels=["<100", "100-150", ">150"]).value_counts().sort_index().to_string())
print()
print(q[["min_dst", "max_kp", "f107", "plasma_dscovr_pct"]].describe().round(1).loc[["min", "50%", "max"]])

48-hour blocks in the record: 1813
quiet candidates: 520

candidates by year:
start
2016    25
2017    56
2018    93
2019    44
2020    93
2021    76
2022    38
2023    40
2024    33
2025     9
2026    13

by F10.7 band (mean over the window):
f107
<100       386
100-150     67
>150        67

     min_dst  max_kp   f107  plasma_dscovr_pct
min    -29.0     0.7   66.0                0.0
50%    -12.0     2.3   75.0               92.0
max      1.0     3.0  329.0              100.0


Pick 100 quiet windows across the solar activity levels and combine them with the storms.

In [4]:
q["band"] = pd.cut(q["f107"], [0, 100, 150, 1000], labels=["<100", "100-150", ">150"])
take = {"<100": 34, "100-150": 33, ">150": 33}
qs = (pd.concat([q[q["band"] == b].sample(n=take[b], random_state=42) for b in take])
        .sort_values("start").reset_index(drop=True))

print("quiet sample:", len(qs))
print("by band:", qs["band"].value_counts().sort_index().to_dict())
print("by year:", qs.groupby(qs["start"].dt.year).size().to_dict())

st = ev.rename(columns={"win_start": "start", "win_end": "end"}).copy()
st["type"] = "storm"
st["cls"] = st["cls"].astype(str)

qq = qs.copy()
qq["type"] = "quiet"
qq["cls"] = "quiet"
qq["min_time"] = [aligned["dst"].loc[a:b].idxmin() for a, b in zip(qq["start"], qq["end"])]
qq["core_start"] = pd.NaT
qq["core_end"] = pd.NaT

keep = ["type", "cls", "start", "end", "min_dst", "min_time", "core_start", "core_end"]
events_all = pd.concat([st[keep], qq[keep]]).sort_values("start").reset_index(drop=True)
events_all["event_id"] = [f"E{i + 1:03d}" for i in range(len(events_all))]
events_all["overlaps_next"] = events_all["start"].shift(-1) <= events_all["end"]
events_all["overlaps_prev"] = events_all["overlaps_next"].shift(1, fill_value=False)

print()
print("combined events:", len(events_all))
print(events_all["cls"].value_counts().to_string())
print("windows overlapping a neighbour:", int(events_all["overlaps_next"].sum()))
print()
print(events_all.head(6).to_string(index=False))

quiet sample: 100
by band: {'<100': 34, '100-150': 33, '>150': 33}
by year: {2016: 4, 2017: 8, 2018: 9, 2019: 2, 2020: 8, 2021: 7, 2022: 17, 2023: 21, 2024: 16, 2025: 3, 2026: 5}

combined events: 302
cls
moderate    171
quiet       100
intense      29
super         2
windows overlapping a neighbour: 29

 type      cls               start                 end  min_dst            min_time          core_start            core_end event_id  overlaps_next  overlaps_prev
quiet    quiet 2016-07-26 00:00:00 2016-07-27 23:00:00      -15 2016-07-26 20:00:00                 NaT                 NaT     E001          False          False
storm moderate 2016-08-02 22:00:00 2016-08-03 22:00:00      -52 2016-08-03 15:00:00 2016-08-03 10:00:00 2016-08-03 16:00:00     E002          False          False
storm moderate 2016-08-23 08:00:00 2016-08-24 07:00:00      -73 2016-08-23 21:00:00 2016-08-23 20:00:00 2016-08-23 22:00:00     E003          False          False
storm moderate 2016-08-31 21:00:00 2016-09

Work out the features for every event (Dst, Bz, speed, pressure, Kp, F10.7 and data quality).

In [5]:
aligned["vbs"] = aligned["v"] * (-aligned["bz"]).clip(lower=0) / 1000       # mV/m
aligned["pdyn"] = 1.67e-6 * aligned["n"] * aligned["v"] ** 2                 # nPa, protons only

H = pd.Timedelta(hours=1)
rows = []
for r in events_all.itertuples():
    A = aligned.loc[r.start:r.end]
    D = aligned.loc[max(r.min_time - 12 * H, r.start):r.min_time]
    rows.append({
        "event_id": r.event_id,
        "hours_start_to_min": (r.min_time - r.start) / H,
        "recovery_hours": (r.end - r.min_time) / H,
        "core_hours": (r.core_end - r.core_start) / H + 1 if r.type == "storm" else np.nan,
        "bz_min": A["bz"].min(),
        "bz_min_time": A["bz"].idxmin(),
        "hours_bz_le_-5": int((A["bz"] <= -5).sum()),
        "vbs_max": A["vbs"].max(),
        "vbs_mean_12h_before_min": D["vbs"].mean(),
        "v_max": A["v"].max(),
        "n_max": A["n"].max(),
        "pdyn_max": A["pdyn"].max(),
        "bt_max": A["bt"].max(),
        "kp_max": A["kp"].max(),
        "ap_max": A["ap"].max(),
        "f107_mean": A["f107_adj"].mean(),
        "f107_spike_in_window": bool(A["f107_spike"].any()),
        "field_dscovr_pct": (A["bz_src"] == "dscovr").mean() * 100,
        "plasma_dscovr_pct": (A["plasma_src"] == "dscovr").mean() * 100,
        "plasma_missing_pct": (A["plasma_src"] == "missing").mean() * 100,
        "dst_min_is_final": bool(aligned["dst_version"].loc[r.min_time] == 20),
        "kp_preliminary": bool((A["kp_D"] < 2).any()),
    })
feat = pd.DataFrame(rows)
event_table = events_all.merge(feat, on="event_id")

cols = ["min_dst", "bz_min", "vbs_max", "v_max", "pdyn_max", "kp_max", "f107_mean", "field_dscovr_pct", "plasma_dscovr_pct"]
print("shape:", event_table.shape)
print()
print("median by class:")
print(event_table.groupby("cls")[cols].median().round(1).reindex(["quiet", "moderate", "intense", "super"]).to_string())

marg = (event_table["type"] == "storm") & (event_table["core_hours"] <= 3) & (event_table["min_dst"] > -60)
print()
print("storm events with core <= 3 h and min Dst above -60:", int(marg.sum()), "of", int((event_table["type"] == "storm").sum()))
print("core hours (storms):")
print(event_table.loc[event_table["type"] == "storm", "core_hours"].describe().round(1).loc[["min", "25%", "50%", "75%", "max"]].to_string())

shape: (302, 32)

median by class:
          min_dst  bz_min  vbs_max  v_max  pdyn_max  kp_max  f107_mean  field_dscovr_pct  plasma_dscovr_pct
cls                                                                                                        
quiet       -12.0    -3.5      1.3  439.0       2.9     2.3      121.9             100.0               76.0
moderate    -61.0    -8.4      4.1  563.0       6.9     5.3      121.3             100.0               81.2
intense    -121.0   -16.1      8.2  557.0      12.8     7.3      155.1             100.0               63.3
super      -369.5   -40.3     27.6  879.5      40.7     8.8      216.8              90.8               29.8

storm events with core <= 3 h and min Dst above -60: 50 of 202
core hours (storms):
min      1.0
25%      3.0
50%      7.0
75%     19.0
max    122.0


Flag the marginal storms, save the event table, and list which events are clean enough to use for evaluation.

In [6]:
event_table["marginal"] = marg.to_numpy()

EVENTS.mkdir(exist_ok=True)
event_table.to_parquet(EVENTS / "event_table.parquet")
event_table.to_csv(EVENTS / "event_table.csv", index=False)
print("saved event_table.parquet and event_table.csv |", event_table.shape)

event_table["clean"] = (~event_table["overlaps_next"] & ~event_table["overlaps_prev"]
                        & (event_table["field_dscovr_pct"] >= 90) & ~event_table["marginal"])

print()
print("clean candidates by class (no overlap, field >= 90% DSCOVR, not marginal):")
print(event_table.groupby("cls")["clean"].agg(clean="sum", total="size").reindex(["quiet", "moderate", "intense", "super"]).to_string())

print()
print("intense and super events with their evaluation properties:")
show = event_table[event_table["cls"].isin(["super", "intense"])].sort_values("min_dst")
print(show[["event_id", "min_time", "min_dst", "kp_max", "field_dscovr_pct", "plasma_dscovr_pct",
            "overlaps_prev", "overlaps_next", "clean"]].round(0).to_string(index=False))

saved event_table.parquet and event_table.csv | (302, 33)

clean candidates by class (no overlap, field >= 90% DSCOVR, not marginal):
          clean  total
cls                   
quiet       100    100
moderate     73    171
intense      17     29
super         0      2

intense and super events with their evaluation properties:
event_id            min_time  min_dst  kp_max  field_dscovr_pct  plasma_dscovr_pct  overlaps_prev  overlaps_next  clean
    E216 2024-05-11 02:00:00     -406     9.0             100.0               29.0          False           True  False
    E242 2024-10-11 01:00:00     -333     9.0              82.0               30.0           True          False  False
    E284 2026-01-20 16:00:00     -236     9.0              73.0               33.0          False           True  False
    E278 2025-11-12 04:00:00     -217     9.0              85.0                8.0          False          False  False
    E158 2023-04-24 05:00:00     -213     8.0             100.0     

/var/folders/4r/925f_39d0qz8x4qc273_15600000gn/T/ipykernel_74567/1936107447.py:19: UserWarning: obj.round has no effect with datetime, timedelta, or period dtypes. Use obj.dt.round(...) instead.
  "overlaps_prev", "overlaps_next", "clean"]].round(0).to_string(index=False))


### 03 event detection findings

- Event table: `data/events/event_table.parquet` / `.csv`, 302 events, 33 columns.
- Storm events: Dst <= -50 nT runs, merged when < 12 h apart, window from 12 h before the core to the end of recovery (Dst > -30 nT, cap 72 h). 202 storms: 171 moderate, 29 intense, 2 super (2024-05-11, 2024-10-11). Window median 30 h, mean 38 h, max 148 h.
- Threshold sensitivity: -50 nT flickers (351 raw runs -> 228 / 202 / 175 at merge gaps 6 / 12 / 24 h); -100 nT is stable (48 -> 31-36); -250 nT gives 2-3 events.
- Quiet windows: 520 candidates (48 h, Dst > -30, Kp <= 3, away from storms, field >= 90% DSCOVR); 100 sampled across F10.7 bands (34 / 33 / 33), every year represented.
- Quiet and storm events separate in the driver features (median Bz min -3.5 vs -8.4 nT; Kp 2.3 vs 5.3). Speed does not separate moderate from intense (563 vs 557 km/s); Bz and vBs do.
- 50 of 202 storms are marginal (core <= 3 h, min Dst above -60 nT); flagged, excluded from evaluation by default.
- 29 storm windows overlap a neighbour; flagged.
- Plasma coverage: 14 of 31 intense/super events have under half of their window's plasma hours from DSCOVR (e.g. 2024-05-11: 29%, 2025-11-12: 8%). Field coverage is high except 2025-09-30 (27%).
- Clean evaluation candidates (no overlap, field >= 90% DSCOVR, not marginal): quiet 100/100, moderate 73/171, intense 17/29, super 0/2. 9 of the intense events also have >= 75% DSCOVR plasma.
- The two super storms both fail `clean`: May 2024 overlaps the next event; October 2024 overlaps the previous one and has 82% field coverage. Proposed: keep May 2024 as the headline case anyway.